# 03 — Repérer plusieurs objets avec YOLO

Le classifieur du notebook précédent proposait une étiquette pour une image entière.
Mais notre photo de rue contient plusieurs choses. Si nous voulons compter des
personnes, l'étiquette « bus » ne suffit pas : il faut aussi savoir **où sont les objets**.

Nous allons lire ce que renvoie un détecteur, le faire travailler sur la photo,
puis examiner ses propositions avant de les compter.
Le fil reste le même : **image → propositions du modèle → vérification → information utile**.

Les **figures de référence** sont enregistrées dans ce cours : elles restent
visibles sans exécuter les cellules. Le code produit vos propres résultats ;
si vous changez une image ou un réglage, ils peuvent différer de ces références.


## 1. Une photo, plusieurs réponses possibles

<img src="../donnees/images/bus.jpg" alt="Photo de rue fournie par Ultralytics : un bus et plusieurs personnes" width="420">

Repérez une personne et le bus : ils n'occupent pas la même partie de la photo.
Un **détecteur d'objets** propose une catégorie et une position pour chaque objet repéré.
Il peut proposer plusieurs fois la même catégorie, ou ne rien proposer du tout.

Cette photo est un exemple de démonstration fourni par
[Ultralytics](https://github.com/ultralytics/ultralytics/tree/v8.3.221/ultralytics/assets).
Elle ne suffit pas à mesurer la fiabilité d'une application.


### Lire le contrat de sortie avant de lancer le modèle

Une **boîte englobante** est un rectangle qui entoure la zone proposée.
Le format `xyxy` donne deux coins : `(x1, y1)` en haut à gauche,
puis `(x2, y2)` en bas à droite. L'origine de l'image est en haut à gauche :
`x` augmente vers la droite, `y` vers le bas.

La **classe** est le nom d'une catégorie, par exemple `person` (« personne »).
Le **score** est un nombre entre 0 et 1 utilisé pour classer les propositions :
plus il est élevé, plus le modèle favorise cette proposition. Il peut se tromper.

![Format d'une détection : coordonnées, classe et score](../ressources/illustrations/03_boite_classe_score.svg)


**Petite vérification.** Avec `[20, 30, 140, 150]`, quelle est la largeur de la boîte ?
Est-ce que `person` permet de connaître le nom de la personne ?


Votre réponse : …


## 2. Choisir le modèle qui répond à cette question

**YOLO** signifie *You Only Look Once* : une famille de modèles qui proposent des
objets et leurs positions à partir de l'image, sans lancer un classifieur séparé
pour chaque découpe possible. Le nom ne signifie pas que le système serait infaillible.

Nous utilisons **YOLO11n**, un modèle compact déjà entraîné. Le `n` désigne la variante
*nano*, la plus petite de cette famille. Ses **poids** sont les nombres appris pendant
cet entraînement ; ils sont fournis dans `modeles/yolo11n.pt`.

Ce détecteur reconnaît les **80 catégories COCO**, dont `person` et `bus`.
C'est un vocabulaire différent des 1 000 catégories du classifieur ImageNet utilisé
précédemment. Un détecteur COCO ne sait donc pas reconnaître n'importe quel objet
ni distinguer toutes les marques de véhicules.

Nous faisons de l'**inférence** : utiliser des poids existants pour calculer une réponse.
Nous ne réentraînons rien. [Modèle et variantes YOLO11](https://docs.ultralytics.com/models/yolo11/),
[catégories COCO](https://docs.ultralytics.com/datasets/detect/coco/).


## 3. Charger l'image et les poids locaux

La première cellule retrouve le dossier du projet et place les réglages logiciels
dans `travaux/config`. Elle vérifie que les poids sont présents **avant** de créer
le modèle : aucun téléchargement de poids ne doit être déclenché par le notebook.

OpenCV lit l'image en **BGR** : bleu, vert, rouge. Nous garderons cette image pour YOLO.
Pour l'afficher avec Matplotlib, nous convertirons seulement l'affichage en **RGB**.


In [ ]:
from pathlib import Path
import os

# Chercher la racine sans dépendre d'un nom de compte ou d'un disque.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'modeles').is_dir() and (p / 'donnees').is_dir())
config = ROOT / 'travaux' / 'config'
config.mkdir(parents=True, exist_ok=True)
os.environ['YOLO_CONFIG_DIR'] = str(config)  # À définir avant l'import Ultralytics.
os.environ['YOLO_OFFLINE'] = 'true'

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO, settings

settings.update({'sync': False})  # Ne pas activer la synchronisation en ligne.
poids = ROOT / 'modeles' / 'yolo11n.pt'
assert poids.is_file(), 'Poids absents : suivre INSTALLATION.txt avant de continuer.'


In [ ]:
# imread lit les pixels du fichier ; None signale une lecture impossible.
image_bgr = cv2.imread(str(ROOT / 'donnees' / 'images' / 'bus.jpg'))
assert image_bgr is not None, 'La photo bus.jpg est introuvable ou illisible.'

hauteur, largeur = image_bgr.shape[:2]  # Deux premières dimensions : lignes, colonnes.
print(f'Image : {largeur} pixels de large × {hauteur} pixels de haut')
plt.figure(figsize=(6, 7))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.title('Photo originale : aucune boîte ajoutée')
plt.axis('off')
plt.show()


`YOLO(...)` construit l'objet Python qui porte notre modèle et ses poids.
`predict(...)` lance le calcul. Ses paramètres rendent nos choix visibles :

- `source=image_bgr` fournit les pixels OpenCV, dans l'ordre BGR attendu ici ;
- `device='cpu'` utilise le processeur du PC ;
- `imgsz=640` fixe la taille cible du traitement interne : la bibliothèque adapte
  l'image et peut ajouter des bordures ; les boîtes renvoyées seront dans les
  coordonnées de la **photo originale** ;
- `conf=0.25` élimine les propositions dont le score est trop faible ; nous allons
  étudier ce filtre juste après avoir vu les résultats ;
- `save=False` ne crée pas de fichiers d'images ; `verbose=False` réduit les messages.

Une liste de résultats est renvoyée : `[0]` prend celui de notre unique image.


In [ ]:
modele = YOLO(str(poids))  # Lire les poids déjà présents, pas un nom à télécharger.
assert modele.task == 'detect', 'Il faut le détecteur, pas le classifieur.'
print('Catégories disponibles :', len(modele.names))
print('Quelques noms :', list(modele.names.values())[:8])

resultat = modele.predict(
    source=image_bgr, imgsz=640, device='cpu', conf=0.25,
    save=False, verbose=False,
)[0]


## 4. Lire une proposition puis l'afficher sur la photo

`resultat.boxes` contient les détections retenues, pas toutes les propositions internes.
`xyxy` porte les coordonnées, `conf` les scores et `cls` les numéros de catégories.
Ces nombres sont stockés dans des **tenseurs**, des tableaux numériques utilisés par
le modèle. `.cpu().numpy()` les rend disponibles comme tableaux NumPy sur le processeur.

`modele.names` associe chaque numéro au nom de la catégorie.
Nous allons construire un `DataFrame` : un tableau Pandas, avec une ligne par détection.
Les positions restent décimales ; l'arrondi ci-dessous sert seulement à la lecture.
Attention au changement de sens : en classification, les 1 000 scores répartissent
la préférence sur les catégories d'une seule image et leur somme vaut 1.
Ici, **chaque boîte possède son score** : les scores de toutes les boîtes ne se
partagent pas un total de 1. Deux personnes peuvent toutes les deux avoir un score élevé.

[Format des résultats](https://docs.ultralytics.com/reference/engine/results/).


In [ ]:
coordonnees = resultat.boxes.xyxy.cpu().numpy()
scores = resultat.boxes.conf.cpu().numpy()
classes = resultat.boxes.cls.cpu().numpy().astype(int)  # Numéros entiers de catégories.

lignes = []
# zip associe la boîte, le score et la classe de la même détection.
for boite, score, numero in zip(coordonnees, scores, classes):
    x1, y1, x2, y2 = boite  # Séparer les quatre coordonnées du rectangle.
    lignes.append([modele.names[numero], float(score), x1, y1, x2, y2])

detections = pd.DataFrame(lignes, columns=['classe', 'score', 'x1', 'y1', 'x2', 'y2'])
display(detections.round(3))  # Même sans détection, les noms de colonnes restent présents.


In [ ]:
# plot dessine les boîtes, les noms de classes et les scores sur une copie BGR.
image_annotee_bgr = resultat.plot(labels=True, conf=True)
plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(image_annotee_bgr, cv2.COLOR_BGR2RGB))
plt.title('Propositions réelles de YOLO11n sur bus.jpg')
plt.axis('off')
plt.show()


**Figure de référence · visible sans exécution**

![La photo de rue annotée avec une boîte bus et quatre boîtes person.](../ressources/illustrations/03_detections_bus.png)

Pour chaque rectangle, reliez sa position, son étiquette et son score à une ligne du tableau. Ce calcul propose un bus et quatre personnes ; les rectangles restent des prédictions à confronter aux pixels.


Comparez une ligne du tableau à sa boîte. La boîte entoure-t-elle bien l'objet annoncé ?
Regardez aussi les bords de l'image : un objet partiellement visible peut être difficile
à reconnaître. **Une boîte dessinée est une proposition, pas une validation humaine.**

Ne cherchez pas à obtenir un nombre « attendu » en déplaçant les réglages :
nous observons d'abord ce que ce modèle produit avec des choix fixés.


## 5. Choisir quelles propositions on accepte de compter

Un **seuil de score** est une valeur minimale que nous choisissons pour garder une
proposition. Par exemple, avec notre règle `score >= 0.50`, une ligne à `0.72` reste
et une ligne à `0.38` est écartée.

Ce filtre n'améliore pas les poids du modèle. Sur une même liste de propositions,
monter le seuil ne peut qu'en retirer. Cela peut réduire les fausses alertes,
mais aussi supprimer des objets correctement détectés.

Un score de `0.80` **ne garantit pas** « 80 % de chances d'avoir raison » sur nos images.
Il faudrait vérifier cette interprétation sur des données représentatives.
Ici, le seuil de démonstration n'est pas une décision de sûreté validée.


In [ ]:
# Même table et mêmes boîtes : seul le seuil de notre règle change.
bilan_seuils = []
for seuil in [0.25, 0.50, 0.85]:
    gardees = detections[detections['score'] >= seuil]
    bilan_seuils.append({'seuil': seuil, 'détections gardées': len(gardees)})
display(pd.DataFrame(bilan_seuils))

etiquettes = [f"{classe} · ligne {i}" for i, classe in enumerate(detections['classe'])]
plt.figure(figsize=(9, 4))
plt.bar(etiquettes, detections['score'], color='#07849a')
plt.axhline(0.50, color='#d05a0b', linestyle='--', label='Seuil 0,50')
plt.axhline(0.85, color='#7d3389', linestyle=':', label='Seuil 0,85')
plt.ylim(0, 1.15)
plt.ylabel('Score de chaque boîte')
plt.legend(loc='upper center', ncol=2)
plt.tight_layout()
plt.show()


**Figure de référence · visible sans exécution**

![Scores des cinq boîtes et lignes horizontales des seuils 0,50 et 0,85.](../ressources/illustrations/03_scores_et_seuil.png)

La barre de la personne la moins bien notée dépasse 0,50 mais reste sous 0,85. Monter le seuil retire cette proposition, sans rien changer à l’image.


Le graphique ne change pas les pixels : la personne écartée reste dans la photo.
La prochaine fonction ajoute un second critère au score : ne compter que la classe `person`.


### Exercice 1 — Compter uniquement les personnes retenues

Complétez `compter_personnes(table, seuil)` : renvoyer le nombre de lignes dont la
classe vaut exactement `'person'` **et** le score est supérieur ou égal au seuil.
Si aucune ligne ne correspond, le résultat doit être `0`. Ne modifiez pas le tableau reçu.

Indices : `table['classe'] == 'person'` produit une condition par ligne ;
`&` combine deux conditions Pandas, chacune entre parenthèses ;
`len(table[condition])` compte les lignes sélectionnées.

Nous comptons des **détections retenues**, pas un nombre de personnes garanti exact.
La table commence déjà au filtre `conf=0.25` : baisser ensuite le seuil de lecture
sous `0.25` ne peut pas ressusciter les propositions absentes.


In [ ]:
def compter_personnes(table, seuil=0.50):
    # À compléter : combiner classe et score, puis compter les lignes retenues.
    return None

nombre_personnes = compter_personnes(detections, seuil=0.50)
if nombre_personnes is None:
    print('Exercice à compléter : le notebook peut continuer.')
else:
    print('Détections person retenues sur la photo :', nombre_personnes)


In [ ]:
# Cas inventés : tester la règle de comptage indépendamment du modèle.
exemple = pd.DataFrame({
    'classe': ['person', 'person', 'person', 'bus'],
    'score': [0.49, 0.50, 0.90, 0.99],
})
avant_test = exemple.copy(deep=True)  # Garder une référence indépendante.
verification = compter_personnes(exemple, 0.50)
if verification is None:
    print('Tests en attente : compléter la fonction ci-dessus.')
else:
    assert exemple.equals(avant_test), 'Ne pas modifier le tableau reçu.'
    assert verification == 2, 'Garder 0.50 et 0.90, mais pas le bus.'
    assert compter_personnes(exemple, 0.95) == 0, 'Aucune personne ne passe ce seuil.'
    assert compter_personnes(exemple.iloc[:0], 0.50) == 0, 'Le tableau vide doit donner 0.'
    print('Tests réussis : classe, frontière du seuil et tableau vide.')


## 6. Pourquoi deux boîtes peuvent-elles entourer la même chose ?

Avant les filtres, un détecteur peut proposer plusieurs rectangles proches pour
un même objet. Le score seul ne permet pas de dire si ces rectangles font doublon.
Nous avons besoin d'une mesure du **recouvrement** : la place qu'ils ont en commun.

L'**IoU**, *Intersection over Union*, est l'aire commune divisée par l'aire totale
couverte par les deux rectangles. L'**intersection** est la zone partagée ;
l'**union** est toute la zone couverte, sans compter deux fois la partie partagée.

![Calcul géométrique de l'IoU, intersection 8 et union 24](../ressources/illustrations/03_iou_surfaces.svg)

Ici, chaque boîte couvre `4 × 4 = 16` unités de surface. La partie commune en couvre `8`.
Donc l'IoU vaut `8 / (16 + 16 − 8)`, environ `0.33`.
Elle vaut `0` sans recouvrement et `1` pour deux boîtes identiques.


In [ ]:
# Refaire exactement le calcul du dessin, sans demander quoi que ce soit au modèle.
aire_a = 4 * 4
aire_b = 4 * 4
intersection = 2 * 4
union = aire_a + aire_b - intersection
iou_exemple = intersection / union
print(f'IoU = {intersection} / {union} = {iou_exemple:.3f}')
assert abs(iou_exemple - 1 / 3) < 1e-9  # Vérifier le résultat malgré les arrondis décimaux.


## 7. Réduire les doublons avec la NMS

La **NMS**, *Non-Maximum Suppression*, garde d'abord la proposition au meilleur score,
puis retire les propositions de la même classe qui la recouvrent trop.
Elle recommence avec les propositions restantes. C'est une règle de tri,
pas un second modèle qui reconnaît les objets.

![Un seuil de score filtre les propositions ; un seuil IoU sert ensuite à réduire les doublons](../ressources/illustrations/03_nms_deux_filtres.svg)

Dans notre **YOLO11**, `predict` réalise déjà ce traitement. Les deux paramètres
ne répondent donc pas à la même question : `conf` concerne le **score d'une proposition** ;
`iou` concerne le **recouvrement entre deux propositions**.

Avec `iou=0.70`, la NMS retire une boîte moins bien notée si son IoU avec une boîte
retenue de même classe dépasse `0.70`. Une valeur IoU plus basse est plus sévère
envers les recouvrements : elle ne signifie pas « plus confiant ».
L'appel suivant reprend les mêmes valeurs que le premier calcul ; il rend
simplement la NMS explicite. Un nombre de boîtes identique n'est **pas** une preuve
que la NMS a retiré un doublon : pour comprendre cette suppression, suivez les
trois propositions de l'illustration ci-dessus.
[Paramètres d'inférence](https://docs.ultralytics.com/modes/predict/),
[algorithme de NMS](https://docs.ultralytics.com/reference/utils/nms/).


In [ ]:
# Même calcul, mais le réglage de suppression des doublons est maintenant explicite.
resultat_explicite = modele.predict(
    source=image_bgr, imgsz=640, device='cpu',
    conf=0.25,                 # Filtrer les scores faibles.
    iou=0.70,                  # Comparer le recouvrement de deux boîtes.
    agnostic_nms=False,         # Ne comparer pour la NMS que des boîtes de même classe.
    save=False, verbose=False,
)[0]
print('Boîtes retenues après ces deux filtres :', len(resultat_explicite.boxes))


La NMS ne garantit pas « une boîte = un objet réel ». Deux personnes très proches
peuvent se recouvrir ; un doublon peut aussi survivre aux filtres.
Les paramètres sont à choisir sur des exemples représentatifs **avec une référence connue**,
et en changeant une chose à la fois pour comprendre son effet.

**Petite vérification.** Si deux rectangles ont exactement la même position,
leur IoU vaut-elle `0` ou `1` ? Est-ce que cela prouve que l'objet annoncé existe ?


Votre réponse : …


## 8. Distinguer ce qui est trouvé, inventé ou manqué

Pour juger le système, il faut une **référence** : des objets vérifiés et leurs boîtes
dessinées à la main, selon des consignes communes. Ce sont des **annotations**.
Nous comparons ensuite les propositions à ces annotations, pas aux scores du modèle.

- **Vrai positif (VP)** : une proposition correspond à un objet de la bonne classe.
- **Faux positif (FP)** : une proposition ne correspond pas à un objet attendu,
  annonce la mauvaise classe, ou fait doublon avec un objet déjà associé.
- **Faux négatif (FN)** : un objet de référence n'a pas été retrouvé.

Pour automatiser cette comparaison, on fixe une règle : par exemple même classe,
`IoU >= 0.50` avec la référence, et une seule association par objet.
**Ce seuil d'évaluation n'est pas le seuil IoU de la NMS** : ici nous comparons une
proposition à une annotation, pas deux propositions entre elles.


### Exercice 2 — Lire une erreur sur un dessin

![Trois personnes de référence ; propositions sur A et B et sur un panneau ; C manquée](../ressources/illustrations/03_vp_fp_fn.svg)

Dans ce **cas inventé**, les propositions sur A et B correspondent bien à leur référence.
Le panneau n'est pas une personne. Comptez les VP, FP et FN.
Puis expliquez pourquoi le score `0.73` affiché sur le panneau ne suffit pas à le valider.

Enfin, si nous supprimons toutes les boîtes pour n'avoir aucune fausse alerte,
combien de personnes de référence seront alors manquées ?


Votre réponse : VP = … ; FP = … ; FN = … ; interprétation = …


Cette distinction prépare deux questions utiles : parmi les propositions,
quelle proportion est juste (**précision**) ? Parmi les objets à retrouver,
quelle proportion a été retrouvée (**rappel**) ?

Un seuil qui retire beaucoup de propositions peut diminuer les fausses alertes
tout en laissant davantage d'objets de côté. Il faut regarder les deux aspects.
Nous ne calculons pas de performance réelle sur `bus.jpg` : le cours ne fournit
pas d'annotations exhaustives vérifiées pour cette photo.


## 9. Essayer une autre image sans changer la question

Nous allons utiliser le **même détecteur, les mêmes poids et les mêmes seuils**
sur une autre photo. Nous changeons seulement les pixels d'entrée.
C'est l'occasion de voir que les catégories possibles ne changent pas avec la photo.
Une personne reste la catégorie `person` : nous ne cherchons pas son identité.


In [ ]:
autre_bgr = cv2.imread(str(ROOT / 'donnees' / 'images' / 'zidane.jpg'))
assert autre_bgr is not None, 'La photo zidane.jpg est introuvable ou illisible.'
autre_resultat = modele.predict(
    source=autre_bgr, imgsz=640, device='cpu', conf=0.25, iou=0.70,
    agnostic_nms=False, save=False, verbose=False,
)[0]

# Voir côte à côte l'entrée inchangée et les propositions réellement calculées.
figure, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cv2.cvtColor(autre_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Autre photo originale')
axes[1].imshow(cv2.cvtColor(autre_resultat.plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title('Même modèle, mêmes réglages')
for axe in axes:
    axe.axis('off')
plt.tight_layout()
plt.show()


**Figure de référence · visible sans exécution**

![Photo de deux personnes sans annotations puis avec les prédictions de YOLO.](../ressources/illustrations/03_autre_scene.png)

Les réglages sont les mêmes que pour le bus. Comparez la version brute et les propositions : deux boîtes `person` et une boîte `tie` (cravate) sont présentes sur ce calcul de référence.


Comparez chaque boîte à la photo originale. S'il manque une boîte, nous pouvons dire
« le modèle n'a pas proposé cette détection avec ces réglages », pas « cet objet est absent ».
Une image contenant de petites silhouettes, une mauvaise lumière ou des objets peu
représentés à l'entraînement peut poser d'autres difficultés.


## 10. Passer de la détection à une petite application

Nous savons maintenant lire une liste de boîtes, la visualiser et appliquer une règle
de comptage. L'étape suivante sera de traiter plusieurs images de la même façon et de
produire un compte-rendu qui conserve le seuil utilisé et ce qui doit être vérifié.

La règle reste la même : **bonne classe ET score suffisant**. Nous choisirons
maintenant `bus` au lieu de `person`, pour retrouver la question du début du cours.
Au lieu de seulement compter, nous conserverons les boîtes sélectionnées afin
de les dessiner et de vérifier le compte-rendu sur l'image.

Notre règle doit aussi accepter le cas **aucune détection** : une liste vide est une
sortie possible, pas forcément une panne. Enfin, compter des boîtes sur plusieurs photos
ne donne pas un nombre de personnes uniques : la même personne peut apparaître plusieurs fois.

[← Classer une image](02_classer_une_image.ipynb) ·
[Suite : construire le compte-rendu →](04_atelier_analyser_des_images.ipynb)
